# Top 10 L6-Level ML Problems
**Track**: Marketing ML Engineer L6 (Product DS + MLE mix)  
**Source**: meta-mle-prep, ML/ads-ranking, ML/LLM, ML/ml-recommenders

Each problem covers:
- Business context and constraints
- Architecture and key ML concepts
- Working code example
- L6-level interview Q&A
- Common pitfalls

---

| # | Problem | Core ML | Serving |
|---|---------|---------|--------|
| 1 | Feed Ranking at Scale | Multi-stage, LambdaMART, NDCG | Online <50ms |
| 2 | Ads CTR Prediction & Ranking | GBDT + IPS + Calibration | Online <10ms |
| 3 | Two-Tower Retrieval | In-batch negatives, FAISS | Online <20ms |
| 4 | Session-Aware Sequential Rec | SASRec (Transformer) | Online <100ms |
| 5 | LTV Optimization via Offline RL | CQL + DR-OPE | Batch (weekly) |
| 6 | LLM Fine-Tuning for Domain Tasks | LoRA/QLoRA, SFT, DPO | Batch (4-hourly) |
| 7 | Uplift Modeling & CATE Estimation | T/X-learner, Qini | Batch |
| 8 | Contextual Bandits | LinUCB, Thompson sampling | Online |
| 9 | ML Monitoring & Distribution Shift | PSI, KL divergence, shadow mode | Continuous |
| 10 | Off-Policy Evaluation & Safe Deployment | DR-OPE, bootstrap CI, guardrails | Pre-launch |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 1: Feed Ranking at Scale

### Business Context
Rank thousands of candidate posts for each user's feed. Goal: maximize engagement (CTR, dwell time, shares) while maintaining content diversity and avoiding filter bubbles.

**Scale**: 500M DAU, 100k posts/user pool, <50ms SLA.

### Architecture — Multi-Stage Pipeline
```
User request
    |
    v
[Stage 1] Candidate Retrieval      ~10k candidates, <10ms
    Two-tower ANN, collaborative filtering, content-based
    |
    v
[Stage 2] Light Ranking             ~1k candidates, <15ms
    GBDT on sparse features, fast inference
    |
    v
[Stage 3] Heavy Ranking             ~200 candidates, <20ms
    Deep neural network, cross-features, user history
    |
    v
[Stage 4] Re-ranking                ~20 items, <5ms
    Diversity (MMR), freshness boost, business rules
```

### Key Concepts

**1. Learning to Rank (LTR) approaches:**
- **Pointwise**: score each item independently (CTR model). No relative order in loss.
- **Pairwise**: penalize inversions between item pairs (RankNet, LambdaRank). Better than pointwise.
- **Listwise**: optimize over the full ranked list (LambdaMART). Closest to NDCG. Industry standard.

**2. NDCG (Normalized Discounted Cumulative Gain):**
```
DCG@k  = Σ relevance_i / log2(position_i + 1)  for i in 1..k
NDCG@k = DCG@k / IDCG@k
```
Position discount: lower-ranked items contribute less. NDCG=1 means perfect ranking.

**3. Position Bias:** Users click top items more regardless of quality. Training on raw clicks conflates relevance with position. Fix: IPS debiasing or position-as-feature (zero at inference).

**4. Feedback Loops:** Deployed model controls what users see → what gets clicked → training data. Over time, popular items get more exposure. Fix: exploration (epsilon-greedy), holdout sets, counterfactual logging.

**5. Diversity (MMR — Maximal Marginal Relevance):**
```
score(i | S) = λ * relevance(i) - (1-λ) * max_{j in S} similarity(i, j)
```
Greedily select items that are relevant but dissimilar to already-selected set S.

In [ ]:
# --- NDCG computation ---
def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain at k."""
    relevances = np.array(relevances[:k], dtype=float)
    if len(relevances) == 0:
        return 0.0
    positions = np.arange(1, len(relevances) + 1)
    return np.sum(relevances / np.log2(positions + 1))

def ndcg_at_k(predicted_order, true_relevances, k):
    """NDCG@k: predicted_order is list of item indices sorted by model score."""
    reordered = [true_relevances[i] for i in predicted_order]
    dcg = dcg_at_k(reordered, k)
    ideal = dcg_at_k(sorted(true_relevances, reverse=True), k)
    return dcg / ideal if ideal > 0 else 0.0

# Example
true_relevances = [3, 0, 2, 1, 3, 0, 1, 2]   # ground truth relevance per item
model_ranking   = [0, 4, 2, 7, 3, 1, 6, 5]    # items sorted by model score (descending)
print(f'NDCG@3: {ndcg_at_k(model_ranking, true_relevances, k=3):.4f}')
print(f'NDCG@5: {ndcg_at_k(model_ranking, true_relevances, k=5):.4f}')

# --- MMR re-ranking for diversity ---
def mmr_rerank(candidates, embeddings, relevance_scores, lam=0.7, k=5):
    """
    Maximal Marginal Relevance.
    candidates: list of item indices
    embeddings: (n_items, d) array
    relevance_scores: (n_items,) array
    lam: trade-off relevance vs diversity (1.0 = pure relevance)
    """
    selected = []
    remaining = list(candidates)

    for _ in range(min(k, len(remaining))):
        if not selected:
            # First item: pick highest relevance
            best = max(remaining, key=lambda i: relevance_scores[i])
        else:
            sel_embs = embeddings[selected]
            def mmr_score(i):
                rel = relevance_scores[i]
                sim = np.max(embeddings[i] @ sel_embs.T)  # cosine if normalized
                return lam * rel - (1 - lam) * sim
            best = max(remaining, key=mmr_score)
        selected.append(best)
        remaining.remove(best)

    return selected

# Synthetic example
n_items, d = 20, 8
embeddings = np.random.randn(n_items, d)
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)  # normalize
relevance_scores = np.random.rand(n_items)

greedy_top5  = list(np.argsort(-relevance_scores)[:5])
diverse_top5 = mmr_rerank(list(range(n_items)), embeddings, relevance_scores, lam=0.6, k=5)

print(f'\nGreedy top-5 (relevance only): {greedy_top5}')
print(f'MMR top-5   (diverse):         {diverse_top5}')
print(f'Avg relevance - greedy: {np.mean(relevance_scores[greedy_top5]):.3f}')
print(f'Avg relevance - MMR:    {np.mean(relevance_scores[diverse_top5]):.3f}')

### L6 Interview Q&A

**Q: Why not just train a pointwise CTR model and sort by it?**  
A: Pointwise models score each item independently — the loss function has no notion of relative order. LambdaMART directly optimizes NDCG by computing gradients from pairwise comparisons weighted by the NDCG gain of swapping each pair. In practice, listwise approaches outperform pointwise by 2-5% NDCG in large-scale feed systems.

**Q: What happens to a ranking model trained on logged click data?**  
A: Position bias inflates CTR for top-ranked items. The model learns to rank items that appeared in position 1 higher — not because they're better, but because they were shown there. Fix: IPS weighting (divide loss by P(shown at position k)), or include position as a feature at training time but zero it out at inference.

**Q: Describe the feedback loop problem and how to address it.**  
A: The deployed model determines what users see → clicks on those items → training data. Items never shown never get clicks, so the model never learns about them. Over time, the item distribution in training data collapses to a narrow set. Fixes: epsilon-greedy exploration (show random items 5% of the time), holdout sets not influenced by the model, counterfactual logging with propensity scores.

### Common Pitfalls
- Using offline NDCG as a proxy for business metrics without online validation
- Forgetting position bias when training on click logs
- Re-ranking at Stage 4 without considering Stage 3 score consistency
- Not logging propensity scores at serving time (needed for IPS debiasing)

---
## Problem 2: Ads CTR Prediction & Ranking

### Business Context
Sponsored search auction. Rank ads using a Generalized Second Price (GSP) auction.  
**Rank score** = bid × pCTR × quality_score. Winner pays the second-highest bid.

**Critical**: pCTR must be **calibrated** (not just a ranking score) because it directly affects the price each advertiser pays. Miscalibration → advertiser churn or lost revenue.

**Scale**: 100k queries/sec, <10ms SLA per query.

### Key Concepts

**1. Position Bias in Ads:**
Ads shown at position 1 get 3-5x more clicks than position 3, regardless of quality.  
Training on raw clicks without correction inflates pCTR for top positions.

**2. Inverse Propensity Scoring (IPS):**
Weight each training example by `1 / P(shown at position k)`.  
Corrects for the fact that high positions are seen more. Unbiased estimator of true CTR.

**3. Calibration:**
Model outputs `P(click) = 0.05` should match empirical click rate of 5%.  
Check with reliability diagram (calibration curve). Fix with Platt scaling (logistic on top of scores) or isotonic regression.

**4. Two-Stage Architecture:**
- Stage 1 (Retrieval): FAISS ANN over ad embeddings, ~500 candidates
- Stage 2 (Ranking): GBDT or deep model on full feature set, <10ms

In [ ]:
# --- IPS-weighted CTR model with Platt scaling calibration ---

def generate_ads_data(n=5000):
    """Synthetic ads click data with position bias."""
    positions = np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.3, 0.25, 0.2, 0.15, 0.1])
    # Propensity: probability of being shown at each position
    propensity_map = {1: 0.30, 2: 0.25, 3: 0.20, 4: 0.15, 5: 0.10}
    propensities = np.array([propensity_map[p] for p in positions])

    # True CTR (position-independent relevance)
    ad_quality = np.random.beta(2, 5, size=n)      # true ad relevance
    position_multiplier = np.array([3.0, 2.0, 1.5, 1.2, 1.0])[positions - 1]
    observed_ctr = ad_quality * position_multiplier
    clicks = (np.random.rand(n) < observed_ctr.clip(0, 1)).astype(int)

    features = np.column_stack([
        ad_quality + np.random.randn(n) * 0.1,   # noisy quality signal
        np.random.randn(n, 4)                     # other features
    ])
    return features, clicks, positions, propensities

X, y, positions, propensities = generate_ads_data()
X_train, X_test, y_train, y_test, pos_train, pos_test, prop_train, prop_test = \
    train_test_split(X, y, positions, propensities, test_size=0.2, random_state=42)

# --- Model 1: Naive (no IPS) ---
naive_model = LogisticRegression(max_iter=1000)
naive_model.fit(X_train, y_train)
naive_preds = naive_model.predict_proba(X_test)[:, 1]

# --- Model 2: IPS-weighted ---
ips_weights = 1.0 / prop_train
ips_weights = np.clip(ips_weights, 1, 20)   # clip to avoid high variance
ips_model = LogisticRegression(max_iter=1000)
ips_model.fit(X_train, y_train, sample_weight=ips_weights)
ips_preds = ips_model.predict_proba(X_test)[:, 1]

# --- Model 3: IPS + Platt scaling calibration ---
calibrated_model = CalibratedClassifierCV(
    LogisticRegression(max_iter=1000), method='sigmoid', cv=3
)
calibrated_model.fit(X_train, y_train, sample_weight=ips_weights)
cal_preds = calibrated_model.predict_proba(X_test)[:, 1]

print('Model Comparison (Test Set):')
print(f'  Naive        AUC: {roc_auc_score(y_test, naive_preds):.4f}  LogLoss: {log_loss(y_test, naive_preds):.4f}')
print(f'  IPS          AUC: {roc_auc_score(y_test, ips_preds):.4f}  LogLoss: {log_loss(y_test, ips_preds):.4f}')
print(f'  IPS+Calibrat AUC: {roc_auc_score(y_test, cal_preds):.4f}  LogLoss: {log_loss(y_test, cal_preds):.4f}')

# --- Expected Calibration Error (ECE) ---
def expected_calibration_error(y_true, y_pred, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_pred >= bins[i]) & (y_pred < bins[i+1])
        if mask.sum() == 0:
            continue
        bin_conf = y_pred[mask].mean()
        bin_acc  = y_true[mask].mean()
        ece += mask.sum() / len(y_true) * abs(bin_conf - bin_acc)
    return ece

print(f'\nECE (lower is better):')
print(f'  Naive:       {expected_calibration_error(y_test, naive_preds):.4f}')
print(f'  IPS+Calibrat:{expected_calibration_error(y_test, cal_preds):.4f}')

### L6 Interview Q&A

**Q: Why does miscalibration hurt revenue in a GSP auction?**  
A: In GSP, the price each winner pays depends on the competitor's bid × pCTR. If pCTR is systematically overestimated, winners overpay → advertiser churn. If underestimated, the platform undercharges → lost revenue. Calibration ensures pCTR equals empirical click probability across score buckets.

**Q: What's the difference between IPS and including position as a feature?**  
A: IPS reweights the loss function to correct for position bias in training. Position-as-feature includes position at training time so the model learns position effects, then zeros it out at inference to get position-independent scores. IPS is unbiased but high variance (clipping helps). Position-as-feature is simpler and often works better in practice. Both are valid; many systems use both.

**Q: Your model has AUC=0.82 but ECE=0.08. Is it production-ready?**  
A: High AUC means good ranking but ECE=0.08 means poor calibration — the model is systematically over/underconfident. For feed ranking alone, it might be fine. For ads, it's not — we need calibrated probabilities to compute fair auction prices. Apply Platt scaling or isotonic regression to fix ECE before deployment.

### Common Pitfalls
- Using ranking AUC as the sole metric when calibration matters for the business logic
- Not clipping IPS weights — extreme weights cause high variance and training instability
- Forgetting to zero out position feature at inference time

---
## Problem 3: Two-Tower Retrieval at Scale

### Business Context
Retrieve top-500 relevant items from a catalog of 5M in <20ms per request. Current matrix factorization scores all items at serving time — 800ms latency, not scalable.

### Architecture
```
User Tower                 Item Tower
user_id emb               item_id emb
history emb (mean pool)   category emb
user attributes           item attributes
      |                         |
      MLP                       MLP
      |                         |
   u ∈ R^d              v ∈ R^d (pre-computed, indexed)
      |                         |
      +--------dot(u,v)----------+
                  |
           Relevance score

Serving: encode user (1 forward pass) → ANN query over item index
```

### Key Concepts

**1. In-batch Negatives (InfoNCE loss):**  
For each positive pair (u_i, v_i), treat all other items in the batch as negatives.  
Efficient — no separate negative mining. With batch size B, each sample gets B-1 negatives.
```
logits = u @ V.T          # (B, B) similarity matrix
labels = eye(B)           # diagonal = positive pairs
loss   = cross_entropy(logits / temperature, labels)
```

**2. Hard Negative Mining:**  
Random negatives are too easy (clearly irrelevant). Hard negatives = items the model ranks highly but user didn't interact with. Improves embedding separation. Typical: 1 positive : 1 random : 1 hard negative.

**3. ANN Index Trade-offs:**
| Index | Build Time | Memory | Recall@100 | Query Latency |
|-------|-----------|--------|------------|---------------|
| FAISS IVF | Fast | Low | ~95% | ~5ms |
| FAISS HNSW | Slow | High | ~99% | ~2ms |
| ScaNN | Medium | Low | ~97% | ~3ms |

HNSW: best recall/latency, graph-based. IVF: clusters items into K centroids, searches nprobe nearest. ScaNN: optimized for inner product (Google's production system).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Tower(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, output_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)  # L2-normalize for cosine similarity

class TwoTowerModel(nn.Module):
    def __init__(self, user_dim=32, item_dim=32, embed_dim=64, temperature=0.07):
        super().__init__()
        self.user_tower = Tower(user_dim, 128, embed_dim)
        self.item_tower = Tower(item_dim, 128, embed_dim)
        self.temperature = temperature

    def forward(self, user_features, item_features):
        u = self.user_tower(user_features)    # (B, d)
        v = self.item_tower(item_features)    # (B, d)
        return u, v

    def infonce_loss(self, u, v):
        """
        In-batch negative loss (InfoNCE).
        u: (B, d) user embeddings
        v: (B, d) item embeddings — row i is the positive item for user i
        """
        B = u.shape[0]
        logits = (u @ v.T) / self.temperature   # (B, B)
        labels = torch.arange(B)                # diagonal = positive pairs
        loss = F.cross_entropy(logits, labels)
        return loss

# Training step simulation
model = TwoTowerModel(user_dim=16, item_dim=16, embed_dim=32)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

B = 64   # batch size
losses = []
for step in range(100):
    user_feats = torch.randn(B, 16)
    item_feats = torch.randn(B, 16)  # row i = positive item for user i
    u, v = model(user_feats, item_feats)
    loss = model.infonce_loss(u, v)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f'Training loss: step 0={losses[0]:.4f} -> step 99={losses[-1]:.4f}')
print(f'Expected minimum (random negatives, B=64): {np.log(B):.4f}')

# Serving: pre-compute item embeddings and do ANN lookup
n_items = 1000
all_item_feats = torch.randn(n_items, 16)
with torch.no_grad():
    item_index = model.item_tower(all_item_feats).numpy()   # (n_items, 32)

query_user = torch.randn(1, 16)
with torch.no_grad():
    user_emb = model.user_tower(query_user).numpy()         # (1, 32)

# Exact NN (in practice, use FAISS)
scores = user_emb @ item_index.T                           # (1, n_items)
top_k  = np.argsort(-scores[0])[:10]
print(f'\nTop-10 retrieved item indices: {top_k}')
print(f'Top-10 scores: {scores[0][top_k].round(3)}')

### L6 Interview Q&A

**Q: Why can't you use a cross-encoder (joint user-item model) for retrieval?**  
A: Cross-encoders process the user and item jointly, so you can't pre-compute item representations. At serving time you'd need one forward pass per item — 5M forward passes is not feasible at <20ms. Two-tower decouples the encodings, pre-computes all item embeddings offline, and only requires one user encoding + ANN lookup at serving time.

**Q: What's the trade-off between temperature in InfoNCE loss?**  
A: Low temperature (e.g. 0.07) sharpens the distribution — model focuses on the hardest negatives in the batch, learns finer-grained distinctions. High temperature makes the loss smoother, easier to optimize but less discriminative. Too low → training instability (gradient exploding). Typical range: 0.05–0.1. Temperature is often a tunable hyperparameter.

**Q: How do you evaluate a retrieval model offline?**  
A: Recall@K — fraction of true positive items found in the top-K retrieved. Target: Recall@500 ≥ 92% so the downstream ranker has sufficient signal. Also check coverage (fraction of catalog ever retrieved) to detect popularity bias in the index.

### Common Pitfalls
- Not L2-normalizing embeddings — cosine similarity requires normalized vectors
- Using batch size too small — InfoNCE needs enough negatives (B ≥ 256 typically)
- Forgetting that item index must be rebuilt when item catalog changes

---
## Problem 4: Session-Aware Sequential Recommendation (SASRec)

### Business Context
User browsing session shows real-time intent that differs from long-term profile. Capture in-session drift to improve recommendations. Target: <100ms SLA, HR@10 ≥ 0.35.

### Architecture
```
Session history: [item_3, item_7, item_1, item_9, ...]  (time-ordered)
                            |
                    Item Embeddings
                            |
             Self-Attention with Causal Mask
             (each position attends only to past)
                            |
                  Last Position Embedding
                            |
               dot(last_emb, candidate_emb)
                            |
                     Ranked candidates
```

### Key Concepts

**1. Causal Masking:** Position i can only attend to positions j ≤ i (no future peeking). Critical for sequential models — without it, the model sees future items during training, leaking information it won't have at inference.

**2. Why Transformers > RNNs for Session Modeling:**
- Attention is parallelizable (LSTM is sequential)
- Direct attention to any position, not just recent — handles long-range dependencies
- Multi-head captures multiple aspects of user interest simultaneously

**3. Cold-Start Graceful Degradation:**
```
if len(history) == 0:  popularity fallback (global top-k)
elif len(history) < 3: item-based CF (nearest neighbors to last item)
else:                  full SASRec pipeline
```

**4. Training: Binary Cross-Entropy with Negative Sampling**  
For each (session, next_item) pair: 1 positive + 100 random negatives. Loss penalizes ranking negatives above the positive.

In [ ]:
class SASRec(nn.Module):
    """
    Self-Attentive Sequential Recommendation.
    Simplified single-layer version for illustration.
    """
    def __init__(self, n_items, d_model=64, n_heads=2, max_seq_len=50, dropout=0.1):
        super().__init__()
        self.item_emb   = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb    = nn.Embedding(max_seq_len, d_model)
        self.attn       = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm       = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)
        self.max_seq    = max_seq_len

    def forward(self, item_seq):
        """
        item_seq: (B, T) tensor of item ids (0 = padding)
        Returns: (B, T, d_model) contextual embeddings
        """
        B, T = item_seq.shape
        positions = torch.arange(T, device=item_seq.device).unsqueeze(0)  # (1, T)

        x = self.item_emb(item_seq) + self.pos_emb(positions)  # (B, T, d)
        x = self.dropout(x)

        # Causal mask: position i cannot attend to j > i
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        # Padding mask: ignore padded positions (item_id=0)
        key_padding_mask = (item_seq == 0)  # (B, T)

        attn_out, _ = self.attn(x, x, x,
                                attn_mask=causal_mask,
                                key_padding_mask=key_padding_mask)
        x = self.norm(x + attn_out)   # residual + layer norm
        return x

    def score_candidates(self, item_seq, candidate_ids):
        """
        Score candidate items given session history.
        candidate_ids: (B, K) candidate item ids
        Returns: (B, K) scores
        """
        seq_emb = self.forward(item_seq)          # (B, T, d)
        last    = seq_emb[:, -1, :]               # (B, d) — use last position
        cand    = self.item_emb(candidate_ids)    # (B, K, d)
        scores  = torch.bmm(cand, last.unsqueeze(-1)).squeeze(-1)  # (B, K)
        return scores

# --- Example usage ---
n_items, B, T, K = 500, 8, 20, 10
model = SASRec(n_items=n_items, d_model=64, n_heads=2, max_seq_len=T)

# Simulated session sequences (0 = padding)
item_seq = torch.randint(1, n_items + 1, (B, T))
item_seq[:, :5] = 0   # first 5 positions are padding

# Candidate items to score
candidates = torch.randint(1, n_items + 1, (B, K))

scores = model.score_candidates(item_seq, candidates)
top1 = scores.argmax(dim=-1)
print(f'Scores shape: {scores.shape}')   # (B, K)
print(f'Top-1 candidate per user: {top1.tolist()}')

# --- Hit Rate@K metric ---
def hit_rate_at_k(scores, ground_truth_idx, k):
    """Fraction of users where the true next item is in top-k."""
    top_k = scores.topk(k, dim=-1).indices  # (B, k)
    hits = (top_k == ground_truth_idx.unsqueeze(-1)).any(dim=-1)
    return hits.float().mean().item()

ground_truth = torch.randint(0, K, (B,))   # index of the true next item in candidates
print(f'\nHR@1: {hit_rate_at_k(scores, ground_truth, 1):.3f}')
print(f'HR@5: {hit_rate_at_k(scores, ground_truth, 5):.3f}')

### L6 Interview Q&A

**Q: Why is the causal mask essential in SASRec?**  
A: Without it, position i can attend to future items j > i. During training, the model sees the answer (future items in the same session), learning a trivial shortcut. At inference, future items don't exist — the model is in a different regime. The causal mask ensures the model only uses past context, making training and inference consistent.

**Q: User's session has only 2 items. What do you do?**  
A: Fall back to item-based CF — find nearest neighbors to the last item by embedding similarity. If 0 items, fall back to global popularity. These fallbacks are critical for cold-start and must be implemented explicitly — never feed a nearly-empty sequence to SASRec as it will produce unreliable embeddings.

**Q: How do you evaluate the session model offline?**  
A: Leave-last-out evaluation: for each session, hold out the last item, train on all others. Measure HR@10 (was true next item in top-10?) and NDCG@10 (how highly was it ranked?). Use chronological splitting — never shuffle sessions randomly, as that leaks future data into training.

### Common Pitfalls
- Shuffling data randomly instead of chronological train/test split — data leakage
- Missing the causal mask — model won't generalize to inference
- Not handling padding correctly in attention (key_padding_mask)

---
## Problem 5: LTV Optimization via Offline Reinforcement Learning

### Business Context
Standard recommenders optimize immediate CTR. This misses long-term value — a user who clicks but churns is worse than one who engages moderately but stays for years. Optimize 12-month Customer Lifetime Value (CLV) using offline RL.

### Why Offline RL?
- **Online RL** (A/B test every policy) is too risky and slow for production
- **Offline RL** learns from logged historical interactions
- **Key challenge**: Q-values for out-of-distribution (OOD) actions are overestimated → model confidently recommends actions never tried before

### MDP Formulation
```
State  s_t:  customer history, profile, behavioral signals, product holdings
Action a_t:  recommendation (item from catalog)
Reward r_t:  revenue - costs + retention_signal
Discount γ:  0.99 (monthly steps)
Objective:   max_π E_π[Σ γ^t r_t]
```

### Conservative Q-Learning (CQL)
Standard Q-learning overestimates unseen actions. CQL adds a penalty:
```
L_CQL = L_Bellman + α * (log Σ_a' exp(Q(s,a')) - E_{a~D}[Q(s,a)])
```
- First term: push Q-values DOWN for all actions (conservative)
- Second term: push Q-values UP for in-distribution actions (preserve accuracy)
- Result: conservative for OOD actions, accurate for seen actions

### Doubly Robust Off-Policy Evaluation (DR-OPE)
Before deploying, estimate the value of the new policy using logged data:
```
V_DR = V_DM + E[(π_e/π_b) * (r - Q_DM(s,a))]
```
- **DM** (Direct Method): use Q-network to estimate value. Low variance, potentially biased.
- **IS** (Importance Sampling): reweight by policy ratio. Unbiased, high variance.
- **DR**: combines both. Correct if either DM or IS is correct — doubly robust.

In [ ]:
# --- CQL Loss and DR-OPE ---

class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    def forward(self, state):
        return self.net(state)   # (B, n_actions)

def cql_loss(q_net, target_net, states, actions, rewards, next_states,
             dones, gamma=0.99, alpha=1.0):
    """
    Conservative Q-Learning loss.
    alpha: CQL regularization strength
    """
    B = states.shape[0]

    # --- Standard Bellman (TD) loss ---
    with torch.no_grad():
        next_q = target_net(next_states).max(dim=1).values    # (B,)
        targets = rewards + gamma * next_q * (1 - dones)      # (B,)

    q_vals = q_net(states)                                    # (B, n_actions)
    q_sa   = q_vals.gather(1, actions.unsqueeze(1)).squeeze() # Q(s,a) for taken actions
    td_loss = F.mse_loss(q_sa, targets)

    # --- CQL penalty ---
    # Push all Q-values down (log-sum-exp term), push in-distribution Q up
    logsumexp_q   = torch.logsumexp(q_vals, dim=1).mean()     # all actions
    in_data_q     = q_sa.mean()                               # logged actions only
    cql_penalty   = logsumexp_q - in_data_q

    total_loss = td_loss + alpha * cql_penalty
    return total_loss, td_loss.item(), cql_penalty.item()

# --- Doubly Robust OPE ---
def doubly_robust_ope(q_net, states, actions, rewards, behavior_probs, eval_probs,
                      max_weight=20.0):
    """
    Estimate value of evaluation policy using logged data.
    behavior_probs: π_b(a|s) — probability of logged action under behavior policy
    eval_probs:     π_e(a|s) — probability under evaluation policy
    """
    with torch.no_grad():
        q_vals = q_net(states)                                  # (B, n_actions)
        q_sa   = q_vals.gather(1, actions.unsqueeze(1)).squeeze()  # Q(s,a)

        # Direct Method: expected value under eval policy
        eval_probs_all = torch.softmax(torch.randn_like(q_vals), dim=1)  # placeholder
        v_dm = (q_vals * eval_probs_all).sum(dim=1).mean()

        # Importance weights (clipped)
        is_weights = (eval_probs / behavior_probs.clamp(min=1e-6)).clamp(max=max_weight)

        # DR correction
        correction = (is_weights * (rewards - q_sa)).mean()
        v_dr = v_dm + correction

    return {'V_DM': v_dm.item(), 'V_DR': v_dr.item()}

# Simulation
state_dim, n_actions, B = 16, 50, 128
q_net     = QNetwork(state_dim, n_actions)
target_net = QNetwork(state_dim, n_actions)
target_net.load_state_dict(q_net.state_dict())
optimizer = torch.optim.Adam(q_net.parameters(), lr=3e-4)

states      = torch.randn(B, state_dim)
actions     = torch.randint(0, n_actions, (B,))
rewards     = torch.randn(B)
next_states = torch.randn(B, state_dim)
dones       = torch.zeros(B)

loss, td, cql_pen = cql_loss(q_net, target_net, states, actions, rewards, next_states, dones)
print(f'CQL Loss: {loss.item():.4f}  (TD: {td:.4f}, CQL penalty: {cql_pen:.4f})')

behavior_probs = torch.rand(B).clamp(0.05, 0.3)
eval_probs     = torch.rand(B).clamp(0.05, 0.3)
ope = doubly_robust_ope(q_net, states, actions, rewards, behavior_probs, eval_probs)
print(f'OPE Estimates: DM={ope["V_DM"]:.4f}  DR={ope["V_DR"]:.4f}')

### L6 Interview Q&A

**Q: What is the fundamental challenge of offline RL vs supervised learning?**  
A: In supervised learning, the training distribution matches the test distribution. In offline RL, the learned policy may select actions that were never (or rarely) taken in the logged data. Q-values for these OOD actions are extrapolated and typically overestimated, leading the agent to confidently recommend actions it has never seen work. CQL penalizes high Q-values for OOD actions, keeping the policy within the support of the logged data.

**Q: CTR drops 8% but CLV increases 3% in the A/B test. How do you explain this to stakeholders?**  
A: This is the expected trade-off. The RL policy sacrifices immediate engagement for long-term value — it recommends items that are slightly less click-worthy but drive deeper engagement, repeat purchases, or retention. Show the unit economics: 3% CLV improvement on a customer base worth $100M annually = $3M incremental revenue. The 8% CTR drop is the cost we pay for that.

**Q: How would you validate the new policy before deploying it?**  
A: DR-OPE on the holdout logged data — if the estimated value improvement is ≥3% with a bootstrapped 95% CI that doesn't include 0, proceed to a small canary (5% traffic). Run canary for 2-4 weeks (long enough to see downstream LTV signal). Gate on: OPE estimate, KL divergence of action distribution from behavior policy ≤ 2 nats, in-distribution rate ≥ 0.85.

### Common Pitfalls
- Using current bid/recommendation as the label (conflates historical policy with optimal)
- Not clipping IS weights — extreme weights (>20) cause OPE variance to explode
- Deploying without OPE and deployment gates — high risk of silent regression

---
## Problem 6: LLM Fine-Tuning for Domain Tasks (LoRA/QLoRA)

### Business Context
Fine-tune an LLM (e.g. LLaMA-3) for bid recommendations in paid search. Given campaign features (CTR, ROAS, budget utilization), the model outputs structured bid adjustment recommendations.

### LoRA (Low-Rank Adaptation)
Instead of updating all W (billions of params), inject trainable low-rank matrices:
```
W' = W_0 + B·A    where B ∈ R^(d×r), A ∈ R^(r×k), r << min(d,k)
```
Only B and A are trained (~0.1-1% of total parameters). W_0 is frozen.

**Rank selection guide:**
- r=4-8: style transfer, formatting tasks
- r=16-32: domain adaptation (typical for bid recs)
- r=64-128: complex reasoning, near full fine-tuning quality

**Alpha scaling:** Set α = 2r. This makes the effective learning rate scale-independent of r.

### QLoRA
Quantize the base model to 4-bit (NF4 quantization), apply LoRA on top. Fits a 65B model on a single 48GB GPU. Training remains in bfloat16 through compute.

### SFT vs DPO
| | SFT (Supervised Fine-Tuning) | DPO (Direct Preference Optimization) |
|---|---|---|
| Data | (prompt, completion) pairs | (prompt, chosen, rejected) triples |
| Process | Standard cross-entropy on completions | Directly optimizes preference without reward model |
| Complexity | Low | Medium (needs preference data) |
| Use when | Clear correct answer exists | Quality requires preference ranking |

In [ ]:
# --- LoRA weight update simulation ---

class LoRALayer(nn.Module):
    """LoRA adapter for a single linear layer."""
    def __init__(self, base_weight, rank=16, alpha=32, dropout=0.05):
        super().__init__()
        d_out, d_in = base_weight.shape
        self.base_weight = nn.Parameter(base_weight, requires_grad=False)  # frozen
        self.A = nn.Parameter(torch.randn(rank, d_in) * 0.01)   # initialized small
        self.B = nn.Parameter(torch.zeros(d_out, rank))          # initialized to 0
        self.scale = alpha / rank                                 # alpha / r scaling
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # W' = W_0 + scale * B @ A
        base_out = F.linear(x, self.base_weight)
        lora_out = self.scale * F.linear(self.dropout(x), self.A.T @ self.B.T)  # B @ A @ x
        return base_out + F.linear(x, self.scale * (self.B @ self.A))

# Inspect parameter counts
d_model, r = 4096, 16
base_params = d_model * d_model
lora_params = r * d_model + d_model * r   # A + B
print(f'Base layer params: {base_params:,}')
print(f'LoRA params (r={r}):  {lora_params:,}')
print(f'Reduction: {lora_params/base_params*100:.1f}% of original')

# Demonstrate LoRA forward pass
W0 = torch.randn(256, 256)
lora = LoRALayer(W0, rank=16, alpha=32)
x = torch.randn(8, 256)
out = lora(x)
print(f'\nLoRA output shape: {out.shape}')

# Count trainable vs frozen
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in lora.parameters() if not p.requires_grad)
print(f'Trainable params: {trainable:,}  Frozen: {frozen:,}')

# --- Structured output evaluation (beyond perplexity) ---
def evaluate_bid_recommendations(predictions, ground_truth):
    """
    Evaluate LLM bid recs beyond perplexity.
    predictions: list of dicts with 'bid_change_pct' and 'reasoning'
    ground_truth: list of dicts with 'optimal_bid_change_pct'
    """
    results = {}

    # 1. JSON validity rate
    valid = sum(1 for p in predictions if isinstance(p, dict) and 'bid_change_pct' in p)
    results['json_validity'] = valid / len(predictions)

    # 2. Directional accuracy (correct increase/decrease/hold)
    correct_direction = sum(
        1 for p, g in zip(predictions, ground_truth)
        if isinstance(p, dict) and np.sign(p.get('bid_change_pct', 0)) == np.sign(g['optimal_bid_change_pct'])
    )
    results['directional_accuracy'] = correct_direction / len(predictions)

    # 3. Magnitude error (MAPE)
    mape_vals = [
        abs(p.get('bid_change_pct', 0) - g['optimal_bid_change_pct']) / (abs(g['optimal_bid_change_pct']) + 1e-6)
        for p, g in zip(predictions, ground_truth)
        if isinstance(p, dict)
    ]
    results['mape'] = np.mean(mape_vals) if mape_vals else 1.0

    return results

# Synthetic evaluation
preds = [{'bid_change_pct': np.random.uniform(-0.3, 0.3), 'reasoning': 'x'} for _ in range(50)]
truth = [{'optimal_bid_change_pct': np.random.uniform(-0.3, 0.3)} for _ in range(50)]
metrics = evaluate_bid_recommendations(preds, truth)
print(f'\nEvaluation metrics: {metrics}')

### L6 Interview Q&A

**Q: When should you fine-tune vs. just prompt-engineer?**  
A: Always measure prompt-engineering baseline first. Fine-tune when: (1) task requires domain knowledge not in pretraining, (2) structured output format must be reliable (JSON schema), (3) latency/cost require a smaller model, (4) privacy prevents sending data to external APIs. If prompt engineering achieves 90% of the target metric, fine-tuning may not be worth the infra cost.

**Q: What is catastrophic forgetting and how do you mitigate it?**  
A: When fine-tuning on domain data, the model overwrites weights used for general language understanding, degrading performance on tasks outside the fine-tuning distribution. Mitigations: LoRA (only adapters are trained, base is frozen), data mixing (blend domain data with general text), early stopping on a held-out general benchmark, Elastic Weight Consolidation (penalizes changing weights important to the original task).

**Q: Why is perplexity insufficient for evaluating bid recommendation quality?**  
A: Perplexity measures how well the model predicts the next token — it's a language quality metric, not a task quality metric. A model can have low perplexity while outputting syntactically plausible but directionally wrong bid recommendations. Measure JSON validity, directional accuracy (increase/decrease/hold), magnitude error (MAPE), and simulate P&L impact using the recommendations on held-out campaign data.

### Common Pitfalls
- Using one prompt template at training, a different one at inference — distribution shift
- Not logging the behavioral policy's π_b(a|s) — needed for OPE later
- Setting rank too high without enough data — overfitting risk

---
## Problem 7: Uplift Modeling & CATE Estimation

### Business Context
A marketing campaign targets all users equally. Some users convert regardless (always-takers), some never convert (never-takers), and some are persuadables (respond to treatment). Targeting only persuadables reduces cost and maximizes incremental lift.

### Key Concepts

**Potential Outcomes Framework:**
```
ATE  = E[Y(1) - Y(0)]                  average treatment effect (all users)
ATT  = E[Y(1) - Y(0) | T=1]            effect on treated users
CATE = E[Y(1) - Y(0) | X=x]            individual-level effect given covariates
```

**Meta-Learners for CATE:**
- **T-Learner**: train separate outcome models μ_0(x), μ_1(x). CATE = μ_1(x) - μ_0(x).
- **S-Learner**: single model with treatment as a feature. CATE = f(x,1) - f(x,0).
- **X-Learner**: uses cross-predictions to impute counterfactuals. Better for imbalanced treatment/control.

**Qini Coefficient:** AUC of the uplift curve — measures how well the model identifies persuadables. A model with Qini=1 perfectly separates persuadables from always-takers and never-takers.

In [ ]:
# --- T-Learner and X-Learner for CATE ---

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier

def generate_uplift_data(n=2000):
    X = np.random.randn(n, 5)
    T = (np.random.rand(n) < 0.5).astype(int)
    # True CATE depends on feature X[:,0]
    true_cate = 0.3 * X[:, 0] + 0.1
    y0 = 0.5 * X[:, 1] + np.random.randn(n) * 0.1    # control outcome
    y1 = y0 + true_cate + np.random.randn(n) * 0.1   # treated outcome
    y  = np.where(T == 1, y1, y0)
    return X, T, y, true_cate

class TLearner:
    def __init__(self):
        self.mu_0 = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.mu_1 = GradientBoostingRegressor(n_estimators=50, random_state=42)

    def fit(self, X, T, y):
        self.mu_0.fit(X[T == 0], y[T == 0])  # control model
        self.mu_1.fit(X[T == 1], y[T == 1])  # treatment model

    def predict_cate(self, X):
        return self.mu_1.predict(X) - self.mu_0.predict(X)

class XLearner:
    """X-Learner: better for imbalanced treatment/control sizes."""
    def __init__(self):
        self.mu_0   = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.mu_1   = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.tau_0  = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.tau_1  = GradientBoostingRegressor(n_estimators=50, random_state=42)
        self.propensity = GradientBoostingClassifier(n_estimators=50, random_state=42)

    def fit(self, X, T, y):
        X0, y0 = X[T == 0], y[T == 0]
        X1, y1 = X[T == 1], y[T == 1]

        self.mu_0.fit(X0, y0)
        self.mu_1.fit(X1, y1)

        # Impute counterfactuals and estimate individual treatment effects
        D0 = self.mu_1.predict(X0) - y0    # imputed effect for control units
        D1 = y1 - self.mu_0.predict(X1)    # imputed effect for treated units

        self.tau_0.fit(X0, D0)
        self.tau_1.fit(X1, D1)
        self.propensity.fit(X, T)

    def predict_cate(self, X):
        e = self.propensity.predict_proba(X)[:, 1]   # propensity score
        return e * self.tau_0.predict(X) + (1 - e) * self.tau_1.predict(X)

X, T, y, true_cate = generate_uplift_data(2000)
X_tr, X_te, T_tr, T_te, y_tr, y_te, tc_tr, tc_te = \
    train_test_split(X, T, y, true_cate, test_size=0.3, random_state=42)

t_learner = TLearner()
t_learner.fit(X_tr, T_tr, y_tr)

x_learner = XLearner()
x_learner.fit(X_tr, T_tr, y_tr)

t_preds = t_learner.predict_cate(X_te)
x_preds = x_learner.predict_cate(X_te)

t_mse = np.mean((t_preds - tc_te) ** 2)
x_mse = np.mean((x_preds - tc_te) ** 2)
print(f'CATE Estimation MSE:')
print(f'  T-Learner: {t_mse:.4f}')
print(f'  X-Learner: {x_mse:.4f}')
print(f'  True CATE range: [{tc_te.min():.2f}, {tc_te.max():.2f}]')

### L6 Interview Q&A

**Q: How is uplift modeling different from A/B testing?**  
A: A/B testing estimates the ATE (average treatment effect across all users). Uplift modeling estimates CATE (treatment effect per user). A/B tells you whether a campaign works on average. Uplift tells you WHO it works for, enabling targeted deployment to persuadables only — reducing cost and avoiding treatment of always-takers (who convert anyway) and never-takers (who don't respond).

**Q: Why is X-Learner better than T-Learner for imbalanced treatment/control?**  
A: T-Learner trains separate models on each arm. With a 10%/90% treatment/control split, the treatment model has very little data and generalizes poorly. X-Learner imputes counterfactuals using the better-trained model from the larger group, then trains the CATE models on the full dataset. The propensity-weighted combination further balances the estimates.

**Q: You deploy the uplift model and target only top-decile users. How do you measure success?**  
A: Measure incremental lift: compare conversion rate of targeted group vs. a holdout of similar users NOT targeted. The Qini coefficient measures ranking quality. Be careful — if you target fewer users, total conversions may drop even if per-user lift increases. Frame as cost-per-incremental-conversion vs. total reach trade-off.

### Common Pitfalls
- Confusing uplift with predicted conversion rate — high converters may not be persuadables
- Using A/B data from non-random assignment — selection bias invalidates CATE
- Not accounting for SUTVA (Stable Unit Treatment Value Assumption) — network effects

---
## Problem 8: Contextual Bandits (LinUCB)

### Business Context
A/B testing is slow and wasteful — it allocates equal traffic to all variants including poor ones. Bandits adaptively allocate more traffic to better-performing options while still exploring.

### Exploration-Exploitation Trade-off
- **Exploit**: choose the action with highest estimated reward (greedy)
- **Explore**: try less-certain actions to gather more information
- **Regret**: cumulative loss vs. always choosing the best action

### Algorithms

| Algorithm | Exploration Strategy | Context-Aware | Best For |
|-----------|---------------------|---------------|----------|
| Epsilon-greedy | Random (ε fraction) | No | Simple, robust baseline |
| UCB1 | Upper confidence bound | No | Stationary rewards |
| Thompson Sampling | Posterior sampling | Yes (with context) | Correlated arms |
| LinUCB | Linear UCB with context | Yes | Marketing personalization |

### LinUCB
Assumes reward is linear in context: r = θ_a^T x + noise.  
Upper confidence bound: UCB_a(x) = θ_a^T x + α √(x^T A_a^{-1} x)  
- First term: exploitation (estimated reward)
- Second term: exploration bonus (uncertainty of estimate for this context)

In [ ]:
class LinUCB:
    """
    LinUCB contextual bandit.
    Each arm has its own ridge regression model.
    """
    def __init__(self, n_arms, context_dim, alpha=1.0):
        self.n_arms = n_arms
        self.alpha  = alpha          # exploration strength
        # Per-arm: A_a (covariance), b_a (reward accumulator)
        self.A = [np.eye(context_dim) for _ in range(n_arms)]
        self.b = [np.zeros(context_dim) for _ in range(n_arms)]

    def select_arm(self, context):
        """Choose arm with highest UCB score."""
        ucb_scores = []
        for a in range(self.n_arms):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]                         # estimated reward weights
            exploit = theta @ context                          # estimated reward
            explore = self.alpha * np.sqrt(context @ A_inv @ context)  # uncertainty
            ucb_scores.append(exploit + explore)
        return int(np.argmax(ucb_scores))

    def update(self, arm, context, reward):
        """Update arm's model with observed reward."""
        self.A[arm] += np.outer(context, context)
        self.b[arm] += reward * context

# Simulation: 3 arms (ad variants), context = user features
n_arms, context_dim, T = 3, 5, 1000
true_theta = np.random.randn(n_arms, context_dim)   # unknown true reward weights

bandit = LinUCB(n_arms, context_dim, alpha=0.5)
cumulative_regret = 0
regrets = []

for t in range(T):
    context = np.random.randn(context_dim)

    # Oracle: best arm for this context
    true_rewards = true_theta @ context
    best_arm = np.argmax(true_rewards)

    # LinUCB selects arm
    chosen = bandit.select_arm(context)

    # Observe reward (noisy)
    reward = true_theta[chosen] @ context + np.random.randn() * 0.1

    # Update model
    bandit.update(chosen, context, reward)

    # Track regret
    cumulative_regret += true_rewards[best_arm] - true_rewards[chosen]
    regrets.append(cumulative_regret)

print(f'LinUCB simulation over {T} steps:')
print(f'  Final cumulative regret: {cumulative_regret:.4f}')
print(f'  Avg regret per step:     {cumulative_regret/T:.4f}')

# Compare arm selection distribution
arm_counts = defaultdict(int)
for t in range(200):  # final 200 steps
    context = np.random.randn(context_dim)
    arm_counts[bandit.select_arm(context)] += 1
print(f'\nArm selection (last 200 steps): {dict(arm_counts)}')

### L6 Interview Q&A

**Q: When would you use a bandit instead of an A/B test?**  
A: Use bandits when: (1) the cost of showing a bad variant is high (revenue loss, user churn), (2) many variants need testing simultaneously, (3) the optimal action changes over time (non-stationary rewards). Use A/B when: you need a clean causal estimate for a business report, regulatory requirement, or the exploration budget is short.

**Q: What's the difference between UCB1 and LinUCB?**  
A: UCB1 is context-free — it estimates a single reward distribution per arm from all observations. LinUCB is context-aware — it models reward as a linear function of user features. LinUCB adapts the exploration bonus based on how uncertain it is about THIS specific user's context, not just overall arm uncertainty. In a marketing setting, LinUCB can learn that arm A is better for young urban users while arm B is better for suburban users.

**Q: How do you handle non-stationary rewards (user preferences change over time)?**  
A: Options: (1) Sliding window — use only recent observations (discard old A and b matrices), (2) Discounted updates — weight recent observations more heavily, (3) Change-point detection — detect when reward distribution shifts and reset the model, (4) Periodic full retraining.

### Common Pitfalls
- Using bandit with too high alpha — over-exploration, converges slowly
- Forgetting that bandits require online serving infrastructure (A matrix must be updated in real time)
- Mixing bandit traffic with A/B holdout traffic — confounds the causal estimate

---
## Problem 9: ML Monitoring & Distribution Shift Detection

### Business Context
A deployed CTR model was trained in January. It's now July. Click rates are dropping. Is this a business problem (users less engaged) or a model problem (distribution shift)? You need automated monitoring to detect and triage.

### Types of Drift

| Type | What Changes | Example | Detection |
|------|-------------|---------|----------|
| Data drift (covariate shift) | P(X) changes, P(Y\|X) same | User demographics shift | PSI on features |
| Concept drift | P(Y\|X) changes, P(X) same | User preferences change | Monitor predictions vs labels |
| Label drift | P(Y) changes | Overall click rate seasonality | Track label distribution |
| Feature drift | One feature goes stale | Partner API stops sending data | Null rates, value ranges |

### Population Stability Index (PSI)
Measures distribution shift between training and serving feature distributions:
```
PSI = Σ (actual% - expected%) * ln(actual% / expected%)

PSI < 0.1:  no significant change
0.1-0.2:    moderate change — investigate
PSI > 0.2:  significant shift — consider retraining
```

In [ ]:
def calculate_psi(expected, actual, n_bins=10, epsilon=1e-4):
    """
    Population Stability Index.
    expected: training distribution
    actual:   serving distribution
    """
    # Use training distribution to define bin edges
    breakpoints = np.quantile(expected, np.linspace(0, 1, n_bins + 1))
    breakpoints[0]  -= 1e-8   # ensure min is included
    breakpoints[-1] += 1e-8   # ensure max is included

    expected_pcts = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pcts   = np.histogram(actual,   bins=breakpoints)[0] / len(actual)

    # Clip to avoid log(0)
    expected_pcts = np.clip(expected_pcts, epsilon, 1)
    actual_pcts   = np.clip(actual_pcts,   epsilon, 1)

    psi = np.sum((actual_pcts - expected_pcts) * np.log(actual_pcts / expected_pcts))
    return psi

# Simulate 3 drift scenarios
np.random.seed(42)
n = 5000

training     = np.random.normal(0, 1, n)          # baseline
no_drift     = np.random.normal(0, 1, n)          # no change
mild_drift   = np.random.normal(0.3, 1.1, n)      # small shift
severe_drift = np.random.normal(1.5, 1.5, n)      # large shift

print('PSI Scores (threshold: 0.1=investigate, 0.2=retrain):')
print(f'  No drift:     {calculate_psi(training, no_drift):.4f}')
print(f'  Mild drift:   {calculate_psi(training, mild_drift):.4f}')
print(f'  Severe drift: {calculate_psi(training, severe_drift):.4f}')

# --- KL Divergence for prediction distribution monitoring ---
def kl_divergence(p, q, n_bins=20, epsilon=1e-8):
    breakpoints = np.linspace(min(p.min(), q.min()), max(p.max(), q.max()), n_bins + 1)
    p_dist = np.histogram(p, bins=breakpoints)[0] / len(p) + epsilon
    q_dist = np.histogram(q, bins=breakpoints)[0] / len(q) + epsilon
    return np.sum(p_dist * np.log(p_dist / q_dist))

train_preds   = np.random.beta(2, 5, n)     # training score distribution
serving_preds = np.random.beta(2, 5, n)     # no shift
shifted_preds = np.random.beta(4, 3, n)     # shifted (model outputs higher scores)

print(f'\nKL Divergence on predictions:')
print(f'  No shift:  {kl_divergence(train_preds, serving_preds):.4f}')
print(f'  Shifted:   {kl_divergence(train_preds, shifted_preds):.4f}')

# --- Monitoring dashboard summary ---
def drift_report(feature_psi_scores, prediction_kl, label_rate_change_pct):
    print('\n=== Drift Monitoring Report ===')
    drifted_features = [(f, v) for f, v in feature_psi_scores.items() if v > 0.1]
    print(f'Features with drift: {len(drifted_features)}/{len(feature_psi_scores)}')
    for f, v in sorted(drifted_features, key=lambda x: -x[1]):
        severity = 'SEVERE' if v > 0.2 else 'MODERATE'
        print(f'  {f}: PSI={v:.3f} [{severity}]')
    print(f'Prediction distribution KL: {prediction_kl:.4f} {"[ALERT]" if prediction_kl > 0.1 else "[OK]"}')
    print(f'Label rate change: {label_rate_change_pct:+.1f}% {"[ALERT]" if abs(label_rate_change_pct) > 10 else "[OK]"}')

drift_report(
    {'age': 0.05, 'session_depth': 0.23, 'device_type': 0.08, 'days_since_last_visit': 0.31},
    prediction_kl=0.07,
    label_rate_change_pct=-12.5
)

### L6 Interview Q&A

**Q: A model's AUC is stable but business metrics are dropping. What do you investigate?**  
A: AUC measures ranking quality independent of calibration. Possible causes: (1) label drift — overall click rate dropped seasonally, model still ranks well but the threshold is wrong; (2) prediction distribution shifted — model is well-calibrated on training data but miscalibrated on current serving traffic; (3) a high-traffic segment the model was accurate on has drifted. Start by checking PSI on features, then KL on prediction distribution, then segment-level performance.

**Q: Explain the difference between data drift and concept drift.**  
A: Data drift (covariate shift): the input feature distribution P(X) changes but the relationship P(Y|X) remains the same. The model would perform fine if it saw the new distribution. Concept drift: P(Y|X) changes — the relationship between features and labels has fundamentally changed. More serious — retraining on old data won't help; you need new labels. In practice, distinguish them by checking whether features or labels (or both) have shifted.

**Q: What is shadow mode and when do you use it?**  
A: Shadow mode runs a new model in parallel with the production model — the new model scores every request but its outputs are NOT shown to users. You collect shadow predictions and compare them to the production model's predictions and (eventually) to outcomes. Use it to: validate a new model's behavior at full production traffic before cutover, detect if a new model would behave very differently, estimate online metrics without risk.

### Common Pitfalls
- Monitoring only overall metrics — segment-level drift can be invisible in aggregate
- Not monitoring null rates and value ranges per feature — silent data pipeline failures
- Setting PSI thresholds too tight — alert fatigue; too loose — misses real issues

---
## Problem 10: Off-Policy Evaluation & Safe Deployment

### Business Context
You've trained a new ranking model. Before rolling it out to 100% of users, you need to estimate its impact using historical logged data (no new A/B test yet) and design a safe staged deployment.

### Off-Policy Evaluation (OPE)
Estimate the value of a new policy π_e using data collected under a different (behavior) policy π_b.

**Three estimators:**

1. **Direct Method (DM)**: use a reward model to predict outcomes under π_e. Low variance but biased if model is wrong.
2. **Importance Sampling (IS)**: reweight logged rewards by π_e(a|s) / π_b(a|s). Unbiased but high variance.
3. **Doubly Robust (DR)**: combines DM and IS. Correct if EITHER the reward model OR IS weights are correct:
```
V_DR = E_D[Q(s,a)] + E_D[(π_e(a|s)/π_b(a|s)) * (r - Q(s,a))]
```

### Staged Deployment Gates
```
OPE check   -> V_DR(π_e) > V_DR(π_b) + δ with 95% CI > 0
Canary 1%   -> monitor for 48h (guardrail metrics: CTR, error rate)
Canary 10%  -> monitor for 1 week (A/B comparison)
Ramp 50%    -> monitor for 1 week
Full 100%   -> rollout complete
Rollback    -> if any guardrail breached, revert to previous version instantly
```

In [ ]:
def doubly_robust_estimator(rewards, actions, contexts,
                             behavior_probs, eval_policy_probs,
                             reward_model_preds, max_weight=20.0):
    """
    Doubly Robust Off-Policy Estimator.

    rewards:            observed rewards (n,)
    actions:            taken actions (n,)
    behavior_probs:     π_b(a|s) for taken actions (n,)
    eval_policy_probs:  π_e(a|s) for taken actions (n,)
    reward_model_preds: Q(s,a) predicted by reward model (n,)
    """
    n = len(rewards)

    # IS weights (clipped)
    is_weights = eval_policy_probs / np.clip(behavior_probs, 1e-6, None)
    is_weights = np.clip(is_weights, 0, max_weight)

    # Direct Method: mean of reward model predictions
    v_dm = reward_model_preds.mean()

    # DR correction: IS-weighted residuals
    residuals = rewards - reward_model_preds
    dr_correction = (is_weights * residuals).mean()

    v_dr = v_dm + dr_correction

    return {'V_DM': v_dm, 'V_DR': v_dr, 'IS_weight_mean': is_weights.mean()}

def bootstrap_confidence_interval(estimator_fn, data, n_bootstrap=1000, ci=0.95):
    """Bootstrap CI for any estimator function."""
    n = len(data['rewards'])
    estimates = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        boot_data = {k: v[idx] for k, v in data.items()}
        est = estimator_fn(**boot_data)
        estimates.append(est['V_DR'])

    alpha = (1 - ci) / 2
    return np.percentile(estimates, [alpha * 100, (1 - alpha) * 100])

# Simulation
np.random.seed(42)
n = 2000

# Logged data under behavior policy
behavior_probs  = np.random.uniform(0.05, 0.4, n)
eval_probs      = np.random.uniform(0.05, 0.4, n)   # new policy probabilities
rewards         = np.random.binomial(1, 0.1, n).astype(float)  # click/no-click
reward_preds    = np.random.uniform(0.05, 0.2, n)   # reward model predictions

# Behavior policy estimate
behavior_est = doubly_robust_estimator(
    rewards, None, None,
    behavior_probs, behavior_probs,   # IS weights = 1 when π_e = π_b
    reward_preds
)

# New policy estimate
eval_est = doubly_robust_estimator(
    rewards, None, None,
    behavior_probs, eval_probs,
    reward_preds
)

print('OPE Results:')
print(f'  Behavior policy V_DR: {behavior_est["V_DR"]:.4f}')
print(f'  Eval policy    V_DR:  {eval_est["V_DR"]:.4f}')
print(f'  Estimated lift: {(eval_est["V_DR"]/behavior_est["V_DR"] - 1)*100:+.1f}%')
print(f'  Avg IS weight:  {eval_est["IS_weight_mean"]:.2f}')

# Bootstrap CI
data = {'rewards': rewards, 'behavior_probs': behavior_probs,
        'eval_policy_probs': eval_probs, 'reward_model_preds': reward_preds}
# Wrapper to match bootstrap API
def est_wrapper(rewards, behavior_probs, eval_policy_probs, reward_model_preds):
    return doubly_robust_estimator(rewards, None, None, behavior_probs, eval_policy_probs, reward_model_preds)

ci_lo, ci_hi = bootstrap_confidence_interval(est_wrapper, data, n_bootstrap=500)
print(f'\n95% Bootstrap CI for V_DR: [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'Deploy recommendation: {"GO" if ci_lo > behavior_est["V_DR"] else "NO-GO (CI includes behavior policy)"}')

### L6 Interview Q&A

**Q: Why is Doubly Robust better than pure Importance Sampling?**  
A: IS is unbiased but has very high variance when π_e and π_b differ significantly — IS weights can be huge, making the estimate unstable. DR adds a control variate (the reward model prediction) that reduces variance while maintaining unbiasedness (the reward model bias is corrected by the IS term). DR is correct if either the IS weights are accurate OR the reward model is accurate — the 'double robustness' property.

**Q: Your bootstrap CI for the new policy is [+0.2%, +4.5%]. Do you deploy?**  
A: The lower bound is positive — the worst-case estimate still shows improvement. Proceed to a 1% canary with a 48-hour monitoring window on guardrail metrics (CTR, p99 latency, error rate). Don't interpret the CI as guaranteeing a real 0.2-4.5% lift — OPE estimates have their own model error. The CI gives directional confidence, not a guaranteed range.

**Q: When should you roll back during a staged deployment?**  
A: Pre-define rollback conditions before deployment: (1) any guardrail metric drops >X% vs. control (e.g. CTR -5%, conversion -3%), (2) error rate increases >0.1%, (3) p99 latency exceeds SLA, (4) unexpected prediction distribution shift (KL > 0.2 vs. baseline). Automated rollback is better than manual — define the conditions in code, not in a runbook.

### Common Pitfalls
- Not logging behavior policy probabilities π_b(a|s) at serving time — makes OPE impossible later
- Not clipping IS weights — one outlier can dominate the estimate
- Deploying directly to 100% without staged rollout — no recovery path if something breaks
- Using OPE as a hard gate rather than one signal among several

---
## Summary: L6 Interview Principles

### 1. Always Clarify Before Modeling
Business metric → proxy metric → model objective. These are three different things. Get alignment on the objective function before discussing algorithms.

### 2. Offline Metrics ≠ Business Metrics
NDCG=0.85 doesn't mean CTR will improve. Always describe the online validation strategy (A/B test design, guardrail metrics) alongside the offline evaluation.

### 3. Distribution Shift Is Always Lurking
Training data ≠ serving data in production. Address it explicitly: IPS for logged feedback, temporal splits for evaluation, PSI monitoring in production.

### 4. Safety First in Deployment
OPE → canary → staged rollout → rollback conditions. Never deploy directly to 100% without a rollback plan.

### 5. Calibration Matters in Business Contexts
Especially for ads (auction pricing) and LTV-based decisions. A well-ranked but miscalibrated model can cost real money.

### 6. Cold Start Is Not an Afterthought
Every ML system must have a graceful degradation strategy for new users and new items. Describe it explicitly.

### 7. Close Every Answer with a Recommendation
At L6, the interviewer expects you to say what YOU would do, not just list options. Always end with: 'I would recommend X because Y, with Z as the key risk to monitor.'

---

| Problem | Key Algorithm | Key Pitfall | Key Metric |
|---------|--------------|-------------|------------|
| Feed Ranking | LambdaMART + MMR | Feedback loops from logged clicks | NDCG@10, online CTR |
| Ads CTR | GBDT + IPS + Platt | Miscalibration in GSP auction | ECE, Log-loss |
| Two-Tower Retrieval | InfoNCE + FAISS | Not L2-normalizing embeddings | Recall@500 |
| SASRec | Transformer + causal mask | Missing causal mask (data leakage) | HR@10, NDCG@10 |
| Offline RL (CLV) | CQL + DR-OPE | OOD Q-value overestimation | DR-OPE estimate, 3-mo LTV |
| LLM Fine-Tuning | LoRA/QLoRA + SFT | Prompt template mismatch train/infer | Directional accuracy, MAPE |
| Uplift Modeling | X-Learner, T-Learner | Confusing uplift with conversion rate | Qini coefficient |
| Contextual Bandits | LinUCB | No IS logging for future OPE | Cumulative regret |
| ML Monitoring | PSI + KL divergence | Monitoring only aggregate metrics | PSI per feature, KL on preds |
| OPE + Deployment | DR-OPE + bootstrap CI | Not logging π_b at serving time | 95% bootstrap CI of V_DR |